# Solutions · 09 · Unsteady reactors, batch heat exchange and runaway

Worked solutions to the exercises of [notebook 09](../notebooks/09_unsteady_reactors_and_runaway.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate
from engreact import energy, kinetics
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
k0, Ea, dH, C_A0, cp_vol = 5.0e7, 68e3, -250e3, 2500.0, 3.6e6

## Exercise 1

A semibatch alternative: feed A over 60 min into the reactor at 320 K with UA/V = 1000 W/(m3 K) (use `multiple.Network` with an energy balance added by hand, or approximate with the batch model by feeding in ten portions). How much lower is the peak temperature than the all-at-once batch?

In [3]:
t = np.linspace(0, 7200, 7201)
batch = energy.batch_with_heat_exchange(k0, Ea, dH, C_A0, 320.0, cp_vol, 1000.0, 300.0, t)
# semibatch approximated as ten portions of A added every 6 min into a reactor that already holds the solvent
def semibatch_portions(n_portions=10, UA_V=1000.0):
    T, C, out = 320.0, 0.0, []
    for i in range(n_portions):
        C += C_A0 / n_portions                                              # add a portion (mixing dilutes only the temperature slightly; volume assumed constant)
        seg = energy.batch_with_heat_exchange(k0, Ea, dH, C, T, cp_vol, UA_V, 300.0, np.linspace(0, 360, 361))
        T, C = seg["T"][-1], seg["C_A"][-1]
        out.append(seg["T"].max())
    return max(out)
print(f"batch, all at once: peak {batch['T_max']:.0f} K; semibatch in ten portions over an hour: peak {semibatch_portions():.0f} K; forty portions over four hours: {semibatch_portions(40):.0f} K")

batch, all at once: peak 491 K; semibatch in ten portions over an hour: peak 430 K; forty portions over four hours: 349 K


Feeding the reactant in portions caps the amount that can react at any moment: ten portions cut the peak by
60 K, forty portions over four hours by far more, because the jacket then keeps up with the heat release.
Semibatch dosing controlled by the temperature (feed only while the reactor is cool enough) is the standard
way to run a fast exothermic reaction safely; the dosing rate is the safety parameter.

## Exercise 2

Relief sizing: assume the reactor is vented when T exceeds 400 K. For the adiabatic batch started at 320 K, compute the rate of heat release (W/m3) at the moment of venting and the fraction of A still unreacted.

In [4]:
t = np.linspace(0, 36000, 36001)
ad = energy.batch_with_heat_exchange(k0, Ea, dH, C_A0, 320.0, cp_vol, 0.0, 300.0, t)
i = int(np.argmax(ad["T"] >= 400.0))
q = (-dH) * kinetics.arrhenius(k0, Ea, ad["T"][i]) * ad["C_A"][i]
print(f"T reaches 400 K after {ad['t'][i]/60:.1f} min with {ad['C_A'][i]/C_A0:.1%} of A unreacted; heat release {q/1e6:.2f} MW/m3 at that moment")

T reaches 400 K after 3.6 min with 50.7% of A unreacted; heat release 27.69 MW/m3 at that moment


By the time the reactor hits the vent temperature more than half the reactant remains and the heat release
is megawatts per cubic metre: the relief system must handle the vapour generated by that power, and a vent
sized for the average rate would be far too small. This is why relief sizing for runaway reactions uses the
DIERS methods and calorimetry, not steady-state heat balances.

## Exercise 3

**Implement it yourself:** write the batch energy balance yourself (two ODEs) and reproduce the peak temperature for UA/V = 1000 W/(m3 K); then add a term for evaporative cooling that caps T at the solvent's boiling point.

In [5]:
def rhs(t_, y, UA_V, T_b=None):
    C, T = y
    r = kinetics.arrhenius(k0, Ea, T) * max(C, 0.0)
    dT = ((-dH) * r - UA_V * (T - 300.0)) / cp_vol
    if T_b is not None and T >= T_b and dT > 0:
        dT = 0.0                                                             # evaporative cooling holds the boiling point
    return [-r, dT]
t = np.linspace(0, 7200, 7201)
sol = integrate.solve_ivp(rhs, (0, 7200), [C_A0, 320.0], t_eval=t, args=(1000.0,), method="LSODA", rtol=1e-8)
lib = energy.batch_with_heat_exchange(k0, Ea, dH, C_A0, 320.0, cp_vol, 1000.0, 300.0, t)
sol_b = integrate.solve_ivp(rhs, (0, 7200), [C_A0, 320.0], t_eval=t, args=(1000.0, 380.0), method="LSODA", rtol=1e-8)
print(f"by hand peak {sol.y[1].max():.1f} K, library {lib['T_max']:.1f} K; with a solvent boiling at 380 K the peak is capped at {sol_b.y[1].max():.0f} K")

by hand peak 491.0 K, library 491.0 K; with a solvent boiling at 380 K the peak is capped at 380 K


The hand-written balance matches the library; adding a boiling-point cap shows why refluxing solvents are
a safety feature: latent heat removes energy far faster than a jacket, and the temperature cannot exceed the
boiling point as long as the condenser keeps up.